# 05 — Calibração A/A (certificado do framework)
Job recorrente (e obrigatório após qualquer mudança de estimador): roda N
experimentos nulos em dados históricos reais e verifica FPR ≈ α, p-values
uniformes e viés ≈ 0. Persiste o certificado em tabela auditável.

In [ ]:
%pip install /Workspace/Shared/supply_experiments/supply_experiments-1.0.2-py3-none-any.whl --quiet --force-reinstall --no-deps
dbutils.library.restartPython()

In [ ]:
from supply_experiments.spark_io import load_city_panel, TABLES
from supply_experiments.calibration.aa import run_aa_calibration
from supply_experiments.estimators.scm import fit_scm
from supply_experiments.estimators.ascm import fit_ascm
from supply_experiments.estimators.sdid import fit_sdid
import supply_experiments

TARGET_ENV = "sandbox"
tables = TABLES[TARGET_ENV]
panel, stats = load_city_panel(spark, "2025-06-01", "2026-06-30")
eligible = [c for c in stats.index if c in panel.cities][:60]

In [ ]:
from datetime import datetime
rows = []
for name, fn in [("scm", fit_scm), ("ascm", fit_ascm), ("sdid", fit_sdid)]:
    aa = run_aa_calibration(panel, eligible, fn, pre_days=120, post_days=35,
                            n_runs=200, n_treated=2, n_donors=15, alpha=0.10)
    print(f"{name}: FPR={aa.fpr:.1%} IC[{aa.fpr_ci[0]:.1%},{aa.fpr_ci[1]:.1%}] "
          f"KS p={aa.ks_p_value:.3f} viés={aa.median_att_bias:+.2%} "
          f"-> {'CALIBRADO' if aa.passed else 'REPROVADO'}")
    rows.append({"run_at": datetime.utcnow(), "estimator": name,
                 "package_version": supply_experiments.__version__,
                 "n_runs": aa.n_runs, "alpha": aa.alpha, "fpr": aa.fpr,
                 "fpr_ci_low": aa.fpr_ci[0], "fpr_ci_high": aa.fpr_ci[1],
                 "ks_p_value": aa.ks_p_value, "median_bias": aa.median_att_bias,
                 "passed": aa.passed})
spark.createDataFrame(rows).write.format("delta").mode("append") \
    .saveAsTable(tables["calibration"])